Antes de rodar, crie um .env na raiz do projeto com:
DB_HOST=localhost
DB_NAME=exercicios_colaborativos
DB_USER=postgres
DB_PASSWORD=sua_senha

In [18]:
# ===================================================
# RF09 — Moderacao de Conteudo (Denuncia)
# ===================================================

import os
from dotenv import load_dotenv

import pandas as pd
import psycopg2 as pg
import sqlalchemy
from sqlalchemy import create_engine
import panel as pn
import matplotlib.pyplot as plt

In [19]:
# Carrega as variaveis do arquivo .env
load_dotenv()

True

In [20]:
# Le as variaveis de ambiente
DB_HOST = os.getenv('DB_HOST')
DB_NAME = os.getenv('DB_NAME')
DB_USER = os.getenv('DB_USER')
DB_PASS = os.getenv('DB_PASSWORD')

In [21]:
# Cria conexao com psycopg2 usando as variaveis carregadas
con = pg.connect(host=DB_HOST, dbname=DB_NAME, user=DB_USER, password=DB_PASS)

In [22]:
# Define a string de conexao para o SQLAlchemy, utilizando as variaveis do .env
# Cria o objeto engine do SQLAlchemy que sera usado para conectar e executar comandos no banco

cnx = f'postgresql://{DB_USER}:{DB_PASS}@{DB_HOST}/{DB_NAME}'

sqlalchemy.create_engine(cnx)

engine = create_engine(cnx)

In [23]:
# Executa a consulta SQL para buscar todos os
# registros da tabela 'Denuncia' no banco PostgreSQL
# e carrega o resultado em um DataFrame do pandas

query = "select * from Denuncia;"
df = pd.read_sql_query(query, cnx)

df

,id_usuario,id_conteudo,motivo,data_hora
0,1,3,Conteúdo duplicado com outra pergunta já exist...,2026-06-10 13:10:56.018428
1,2,4,Texto sem contexto suficiente para ser respond...,2026-06-10 13:10:56.018428
2,3,5,Pergunta fora da disciplina selecionada.,2026-06-10 13:10:56.018428
3,6,7,Resposta com informações incorretas.,2026-06-10 13:10:56.018428
4,7,8,Linguagem inadequada no texto.,2026-06-10 13:10:56.018428
5,9,9,Conteúdo copiado de outra fonte sem referência.,2026-06-10 13:10:56.018428
6,1,10,Pergunta repetida já respondida anteriormente.,2026-06-10 13:10:56.018428
7,2,11,Resposta incompleta e enganosa.,2026-06-10 13:10:56.018428
8,3,13,Imagem anexada com conteúdo inapropriado.,2026-06-10 13:10:56.018428
9,6,15,Resposta não responde à pergunta feita.,2026-06-10 13:10:56.018428


In [24]:
# Inicializa as extensoes do Panel necessarias para exibir tabelas
# interativas (Tabulator) e notificacoes na interface grafica

pn.extension()
pn.extension('tabulator')
pn.extension(notifications=True)

BokehModel(combine_events=True, render_bundle={'docs_json': {'2aaf78a8-f6f2-4dd2-8439-f7dce8e1916b': {'version…

In [25]:
# campos de texto

# declare esta variavel para usar na consulta de campos em branco
flag = ''

# Cria widgets interativos para o usuario inserir ou selecionar dados.
# id_usuario e id_conteudo sao carregados do banco via dropdown,
# evitando referencias a registros inexistentes.

def carregar_opcoes_usuario():
    df = pd.read_sql_query(
        "SELECT id_usuario, p_nome, sobrenome FROM Usuario ORDER BY id_usuario", cnx
    )
    opcoes = {"— Selecione —": None}
    for _, r in df.iterrows():
        opcoes[f"{r['p_nome']} {r['sobrenome']} (#{int(r['id_usuario'])})"] = int(r['id_usuario'])
    return opcoes

def carregar_opcoes_conteudo():
    df = pd.read_sql_query(
        "SELECT id_conteudo, LEFT(corpo_do_texto, 50) AS resumo FROM Conteudo ORDER BY id_conteudo", cnx
    )
    opcoes = {"— Selecione —": None}
    for _, r in df.iterrows():
        opcoes[f"#{int(r['id_conteudo'])} - {r['resumo']}"] = int(r['id_conteudo'])
    return opcoes

usuario_denuncia = pn.widgets.Select(
    name="Usuario que Denuncia *",
    options=carregar_opcoes_usuario()
)

conteudo_denuncia = pn.widgets.Select(
    name="Conteudo Denunciado *",
    options=carregar_opcoes_conteudo()
)

motivo = pn.widgets.TextAreaInput(
    name="Motivo da Denuncia *",
    value='',
    placeholder='Descreva o motivo da denuncia...',
    disabled=False
)

filtro_busca = pn.widgets.TextInput(
    name="Buscar (trecho do motivo)",
    value='',
    placeholder='Digite para filtrar...',
    disabled=False
)

# Para excluir: chave primaria composta (id_usuario + id_conteudo)
id_usuario_excluir = pn.widgets.IntInput(
    name="ID do Usuario (para Excluir)",
    value=0
)

id_conteudo_excluir = pn.widgets.IntInput(
    name="ID do Conteudo (para Excluir)",
    value=0
)

In [26]:
# Cria quatro botoes para as acoes principais da aplicacao CRUD:
# Consultar, Inserir, Excluir e Atualizar registros no banco de dados

buttonConsultar = pn.widgets.Button(name='Consultar', button_type='default')

buttonInserir = pn.widgets.Button(name='Inserir', button_type='default')

buttonExcluir = pn.widgets.Button(name='Excluir', button_type='default')

buttonAtualizar = pn.widgets.Button(name='Atualizar', button_type='default')

In [27]:
def queryAll():
    """
    Consulta todos os registros da tabela 'Denuncia' no banco de dados,
    com o nome do usuario denunciante e o resumo do conteudo denunciado,
    e retorna um widget Tabulator para exibicao interativa dos dados.

    Returns:
        pn.widgets.Tabulator: Widget que exibe a tabela com todos os dados da tabela 'Denuncia'.
    """
    query = """
        SELECT d.id_usuario, u.p_nome || ' ' || u.sobrenome AS denunciante,
               d.id_conteudo, LEFT(c.corpo_do_texto, 60) AS conteudo_denunciado,
               d.motivo, d.data_hora
        FROM Denuncia d
        JOIN Usuario u ON d.id_usuario = u.id_usuario
        JOIN Conteudo c ON d.id_conteudo = c.id_conteudo
        ORDER BY d.data_hora DESC
    """
    df = pd.read_sql_query(query, cnx)
    return pn.widgets.Tabulator(df)


def limpar_campos():
    """
    Limpa todos os campos do formulario, deixando-os prontos para uma nova operacao.
    """
    usuario_denuncia.value = None
    conteudo_denuncia.value = None
    motivo.value = ''
    filtro_busca.value = ''
    id_usuario_excluir.value = 0
    id_conteudo_excluir.value = 0


# consultar
# neste exemplo o metodo de consulta usa o dataframe do pandas como retorno. Note que a flag e usada para ignorar quando um
# campo for null (condicao e sempre verdadeira).
def on_consultar():
    """
    Consulta registros na tabela 'Denuncia' filtrando por um trecho do motivo
    informado. Se o campo estiver vazio, retorna todos os registros.

    Returns:
        pn.widgets.Tabulator ou pn.pane.Alert: Tabela com os dados encontrados ou alerta em caso de erro.
    """
    try:
        query = f"""
            SELECT d.id_usuario, u.p_nome || ' ' || u.sobrenome AS denunciante,
                   d.id_conteudo, LEFT(c.corpo_do_texto, 60) AS conteudo_denunciado,
                   d.motivo, d.data_hora
            FROM Denuncia d
            JOIN Usuario u ON d.id_usuario = u.id_usuario
            JOIN Conteudo c ON d.id_conteudo = c.id_conteudo
            WHERE ('{filtro_busca.value_input}'='{flag}' OR d.motivo ILIKE '%{filtro_busca.value_input}%')
            ORDER BY d.data_hora DESC
        """
        df = pd.read_sql_query(query, cnx)
        table = pn.widgets.Tabulator(df)
        return table
    except:
        return pn.pane.Alert('Não foi possível consultar!')


def on_inserir():
    """
    Insere uma nova denuncia na tabela 'Denuncia', vinculada a um usuario
    e a um conteudo existentes. A chave primaria e composta (id_usuario, id_conteudo),
    entao o mesmo usuario nao pode denunciar o mesmo conteudo duas vezes.
    Ao final, limpa o formulario automaticamente.

    Returns:
        pn.widgets.Tabulator ou pn.pane.Alert: Tabela atualizada ou alerta em caso de erro.
    """
    try:
        if not usuario_denuncia.value or not conteudo_denuncia.value:
            return pn.pane.Alert('Selecione o usuario e o conteudo denunciado.')

        cursor = con.cursor()
        cursor.execute(
            "INSERT INTO Denuncia (id_usuario, id_conteudo, motivo) VALUES (%s, %s, %s)",
            (usuario_denuncia.value, conteudo_denuncia.value, motivo.value_input)
        )
        cursor.query
        con.commit()
        pn.state.notifications.success('Denuncia registrada com sucesso!')
        limpar_campos()
        return queryAll()
    except Exception as e:
        cursor.execute("ROLLBACK")
        cursor.close()
        return pn.pane.Alert(f'Não foi possível inserir: {str(e)}')


def on_atualizar():
    """
    Atualiza o motivo da denuncia identificada pelo par (id_usuario, id_conteudo)
    informado nos campos de exclusao/atualizacao.
    Ao final, limpa o formulario automaticamente.

    Returns:
        pn.widgets.Tabulator ou pn.pane.Alert: Tabela atualizada ou alerta em caso de erro.
    """
    try:
        if not id_usuario_excluir.value or not id_conteudo_excluir.value:
            return pn.pane.Alert('Informe o ID do usuario e o ID do conteudo para atualizar.')

        cursor = con.cursor()
        cursor.execute(
            "UPDATE Denuncia SET motivo = %s WHERE id_usuario = %s AND id_conteudo = %s",
            (motivo.value_input, id_usuario_excluir.value, id_conteudo_excluir.value)
        )
        cursor.query
        con.commit()
        pn.state.notifications.success('Denuncia atualizada com sucesso!')
        limpar_campos()
        return queryAll()
    except Exception as e:
        cursor.execute("ROLLBACK")
        cursor.close()
        return pn.pane.Alert(f'Não foi possível atualizar: {str(e)}')


def on_excluir():
    """
    Exclui o registro da tabela 'Denuncia' identificado pelo par
    (id_usuario, id_conteudo) informado nos campos correspondentes.
    Ao final, limpa o formulario automaticamente.

    Returns:
        pn.widgets.Tabulator ou pn.pane.Alert: Tabela atualizada ou alerta em caso de erro.
    """
    try:
        if not id_usuario_excluir.value or not id_conteudo_excluir.value:
            return pn.pane.Alert('Informe o ID do usuario e o ID do conteudo para excluir.')

        cursor = con.cursor()
        cursor.execute(
            "DELETE FROM Denuncia WHERE id_usuario = %s AND id_conteudo = %s",
            (id_usuario_excluir.value, id_conteudo_excluir.value)
        )
        rows_deleted = cursor.rowcount
        con.commit()
        pn.state.notifications.success('Denuncia removida com sucesso!')
        limpar_campos()
        return queryAll()
    except:
        cursor.execute("ROLLBACK")
        cursor.close()
        return pn.pane.Alert('Não foi possível excluir!')

In [28]:
# Funcao que chama a acao correta (consultar, inserir, atualizar ou excluir)
# dependendo do botao que foi clicado (representado pelos parametros booleanos)

def table_creator(cons, ins, atu, exc):
    if cons:
        return on_consultar()
    if ins:
        return on_inserir()
    if atu:
        return on_atualizar()
    if exc:
        return on_excluir()

In [29]:
# Cria uma ligacao interativa (bind) entre os botoes e a funcao que executa a acao correspondente,
# atualizando a tabela na interface sempre que algum botao for clicado.

interactive_table = pn.bind(table_creator, buttonConsultar, buttonInserir, buttonAtualizar, buttonExcluir)

In [30]:
def criar_graficos_pandas(event=None):
    query_usuario = """
        SELECT u.p_nome || ' ' || u.sobrenome AS usuario, COUNT(*) as qtd
        FROM Denuncia d
        JOIN Usuario u ON d.id_usuario = u.id_usuario
        GROUP BY u.p_nome, u.sobrenome
        ORDER BY qtd DESC
        LIMIT 10
    """
    query_conteudo = """
        SELECT LEFT(c.corpo_do_texto, 40) AS conteudo, COUNT(*) as qtd
        FROM Denuncia d
        JOIN Conteudo c ON d.id_conteudo = c.id_conteudo
        GROUP BY c.corpo_do_texto
        ORDER BY qtd DESC
        LIMIT 10
    """

    def_usuario = pd.read_sql(query_usuario, engine)

    def_conteudo = pd.read_sql(query_conteudo, engine)

    fig1, ax1 = plt.subplots(figsize=(6, 5))

    fig2, ax2 = plt.subplots(figsize=(6, 5))

    if not def_usuario.empty:
        def_usuario.set_index('usuario').plot(
            kind='bar',
            y='qtd',
            ax=ax1,
            color='#d9534f',
            rot=25,
            legend=False,
            title='Usuarios que mais Denunciaram'
        )
        ax1.set_ylabel('Quantidade de Denuncias')
    else:
        ax1.text(0.5, 0.5, "Sem dados", ha='center')

    if not def_conteudo.empty:
        def_conteudo.set_index('conteudo').plot(
            kind='barh',
            y='qtd',
            ax=ax2,
            color='#f0ad4e',
            legend=False,
            title='Conteudos mais Denunciados'
        )
        ax2.set_xlabel('Quantidade de Denuncias')
        ax2.invert_yaxis()
    else:
        ax2.text(0.5, 0.5, "Sem dados", ha='center')

    plt.close(fig1)
    plt.close(fig2)

    return pn.Column(
        pn.pane.Matplotlib(fig1, tight=True),
        pn.pane.Matplotlib(fig2, tight=True),
        name="Graficos"
    )

In [31]:
# Monta o layout da interface com Panel:
# - Coluna esquerda com o titulo, os campos de entrada e os botoes de acao
# - Coluna direita com a tabela interativa que mostra os dados do banco
# O metodo `.servable()` permite que essa interface seja exibida ao rodar o Panel server

layout_crud = pn.Row(
    pn.Column(
        'Denuncia CRUD',
        usuario_denuncia, conteudo_denuncia, motivo, filtro_busca,
        id_usuario_excluir, id_conteudo_excluir,
        pn.Row(buttonConsultar, buttonInserir),
        pn.Row(buttonAtualizar, buttonExcluir),
    ),
    pn.Column(interactive_table),
    name="Gerenciamento"
)

layout_dashboard = pn.bind(criar_graficos_pandas, buttonInserir)

aba_principal = pn.Tabs(
    layout_crud,
    ("Dashboard (Pandas)", layout_dashboard)
)

aba_principal.servable()

BokehModel(combine_events=True, render_bundle={'docs_json': {'93dd3da5-fbc1-4a66-989f-960eaf1db5f6': {'version…